# LangChain Prompting – Setup

**Goal:** Run simple prompts using the Groq chat model.

### 1. Install packages
Run this cell once in your notebook. If you install packages while Jupyter is open, restart the kernel if imports fail.

In [ ]:
%pip install -q -U langchain langchain-core langchain-groq python-dotenv

### 2. Configure your Groq API key with a `.env` file

For local development, use `python-dotenv` to load the API key from a `.env` file. This keeps credentials out of notebook code and makes the setup repeatable.

1. In the same project folder as these notebooks, create a file named `.env`.
2. Add this line, replacing the placeholder with your own Groq API key:

   `GROQ_API_KEY=your_actual_groq_api_key`

3. Keep `.env` private. The included `.gitignore` excludes it from Git.
4. Run the next setup cell before initializing the model.

Do not commit, upload, or share your real `.env` file. The included `.env.example` is only a template and contains no real credential.


In [1]:
# Load credentials from the local .env file.
import os
from dotenv import load_dotenv

load_dotenv()  # Reads .env from the current project directory (or parent directories).

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
# if not GROQ_API_KEY:
#     raise ValueError(
#         "GROQ_API_KEY was not found. Create a .env file beside the notebooks "
#         "with: GROQ_API_KEY=your_actual_groq_api_key"
#     )

print("Groq API key loaded successfully.")


Groq API key loaded successfully.


### 3. Create the chat model

`ChatGroq` connects LangChain to a Groq-hosted chat model. Model names can change, so use a model currently available in your Groq account.

In [2]:
from langchain_groq import ChatGroq

# Keep the model name configurable because provider model availability can change.
MODEL_NAME = "openai/gpt-oss-120b"

llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0,
    api_key=GROQ_API_KEY
)
print("Groq chat model initialized.")


Groq chat model initialized.


> If Groq reports that the model name is unavailable or decommissioned, replace `model` with a currently supported chat model from your Groq Console.

> **Security and setup:** The original hardcoded API keys have been removed. Run the secure key-entry cell and model setup cell before running model examples. Use a newly generated key; revoke any key that was previously exposed. The model name shown is the one used in the source notebook; change `MODEL_NAME` if it is no longer available in your Groq account.

# Day 2 – PromptTemplate

## Learning objectives
- Understand reusable prompt templates.
- Use input variables.
- Use multiple variables.
- Format prompts and create partial templates.

## 1. Why PromptTemplate?

Without a template, you may repeatedly write almost the same prompt. A template lets you keep the instruction fixed and change only the input values.

In [3]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template(
    "Explain {topic} in simple English with one example."
)

prompt_value = template.format(topic="Python functions")
print(prompt_value)

Explain Python functions in simple English with one example.


## 2. Send a formatted prompt to Groq

In [4]:
prompt_text = template.format(topic="AWS EC2")
response = llm.invoke(prompt_text)
print(response.content)

**What is AWS EC2?**  

Amazon Elastic Compute Cloud (EC2) is a service that lets you run a computer (a “virtual server”) in Amazon’s data centers.  
Think of it as renting a computer that lives in the cloud instead of buying a physical machine and keeping it in your office.  

- **You pick the size you need** – small, medium, big, with more CPU, memory, storage, etc.  
- **You only pay for the time it runs** (by the hour or second).  
- **You can start, stop, or delete it whenever you want** – just like turning a laptop on or off.  
- **Amazon takes care of the hardware, power, cooling, and networking**; you only manage the software you install on the server.

---

### Simple real‑world example

**Goal:** Host a personal blog that anyone on the internet can read.

1. **Create an EC2 instance**  
   - In the AWS console you click “Launch Instance”.  
   - Choose an Amazon Machine Image (AMI) that already has Linux and the web‑server software (e.g., an “Ubuntu + Apache” AMI).  
   - Pic

## 3. Multiple input variables

In [5]:
template = PromptTemplate.from_template(
    "You are a {role}. Explain {topic} to a {audience}. Give {count} examples."
)

prompt_text = template.format(
    role="Python trainer",
    topic="for loops",
    audience="beginner",
    count=2
)

print(prompt_text)
print("\nMODEL ANSWER:")
print(llm.invoke(prompt_text).content)

You are a Python trainer. Explain for loops to a beginner. Give 2 examples.

MODEL ANSWER:
## What a **`for` loop** does  

A `for` loop lets you repeat (or *iterate*) a block of code many times, automatically moving through a collection of items—like the numbers 0‑9, the words in a list, or the characters in a string.  

Think of it as a **conveyor belt**:

1. The loop picks the **first** item.  
2. It runs the indented code block using that item.  
3. Then it moves to the **next** item, runs the block again, and so on—until there are no items left.

### Basic syntax

```python
for <variable> in <iterable>:
    # code that runs for each item
    # (must be indented)
```

- **`<variable>`** is a name you choose; it will hold the current item each time through the loop.  
- **`<iterable>`** is anything you can “loop over”: a list, a tuple, a string, a range of numbers, etc.  
- The **colon (`:`)** ends the `for` line, and everything that is **indented** underneath belongs to the loop bo

## 4. Use a template repeatedly

In [6]:
template = PromptTemplate.from_template(
    "Create a short {type} about {subject} for {audience}."
)

topics = ["Python lists", "SQL databases", "Docker containers"]

for topic in topics:
    prompt_text = template.format(
        type="lesson introduction",
        subject=topic,
        audience="new IT students"
    )
    print("\nTOPIC:", topic)
    print(llm.invoke(prompt_text).content)


TOPIC: Python lists
## 📚 Lesson Introduction: Python Lists  
**Audience:** New IT students (no prior programming experience required)  
**Duration:** 10‑15 minutes (lecture + quick hands‑on)

---

### 1️⃣ Why Lists Matter
- **Collections:** Most real‑world data comes in groups – a shopping cart, a playlist, sensor readings, etc.  
- **Flexibility:** Lists let you store *any* mix of values (numbers, strings, even other lists) and change them on the fly.  
- **Foundation:** Mastering lists opens the door to loops, functions, data analysis, and almost every Python library.

---

### 2️⃣ Learning Objectives
By the end of this short lesson you will be able to:

| Objective | What it looks like in code |
|-----------|----------------------------|
| **Create** a list | `my_list = [1, 2, 3]` |
| **Access** items by index | `my_list[0]` → `1` |
| **Modify** elements | `my_list[1] = 99` |
| **Add / remove** items | `my_list.append(4)` / `my_list.pop()` |
| **Iterate** over a list | `for item in

## 5. format() versus format_prompt()

In [7]:
template = PromptTemplate.from_template("What is {concept}?")

# format() returns a normal Python string
text_prompt = template.format(concept="API")
print(type(text_prompt))
print(text_prompt)

# format_prompt() returns a PromptValue object
prompt_value = template.format_prompt(concept="API")
print(type(prompt_value))
print(prompt_value.to_string())

<class 'str'>
What is API?
<class 'langchain_core.prompt_values.StringPromptValue'>
What is API?


## 6. Partial variables

In [8]:
template = PromptTemplate.from_template(
    "You are a {role}. Explain {topic} in {style} language."
)

trainer_template = template.partial(
    role="Cloud DevOps trainer",
    style="simple"
)

print(trainer_template.format(topic="CI/CD"))
print(llm.invoke(trainer_template.format(topic="CI/CD")).content)

You are a Cloud DevOps trainer. Explain CI/CD in simple language.
## CI/CD in Plain English  

Think of building a software product like baking a cake:

| Step | Traditional (manual) way | CI/CD (automated) way |
|------|--------------------------|-----------------------|
| **Write the recipe** | A developer writes code on their laptop. | The code lives in a shared “recipe book” (a version‑control system like Git). |
| **Gather ingredients** | Every time someone wants to test the cake, they have to manually copy the code, install libraries, set up a database, etc. | A **CI** (Continuous Integration) server automatically pulls the latest code, fetches the right libraries, and prepares a clean environment every time a change is made. |
| **Mix & bake** | A developer runs the build and tests on their own machine. If they forget a step, the cake may turn out wrong, and the problem is discovered late. | The CI server **builds** the code (compiles, packages) and runs **automated tests** inst

## 7. Common mistakes
- Misspelling a variable name.
- Forgetting to provide a required variable.
- Using `{}` for text that should be treated literally without escaping it.
- Expecting a template to call the model automatically. A template only creates the prompt; `llm.invoke()` sends it.

## Student practice
Create a template that accepts `student_name`, `course`, and `duration`, then asks Groq to generate a welcome message.

## 8. Input variables, required inputs, and validation

A template variable is a named placeholder such as `{topic}`. The value is supplied when formatting or invoking the template. `input_variables` tells you which values are required.

**Why it is useful:** Validation catches missing or misspelled inputs early, before you send an incomplete prompt to a model.

In [9]:
from langchain_core.prompts import PromptTemplate

validated_template = PromptTemplate(
    template="Explain {topic} to a {audience}.",
    input_variables=["topic", "audience"],
    validate_template=True
)

print("Required inputs:", validated_template.input_variables)
print(validated_template.format(topic="Docker", audience="beginner"))

# Uncomment to see a missing-variable error:
# validated_template.format(topic="Docker")

Required inputs: ['audience', 'topic']
Explain Docker to a beginner.


## 9. Reuse a template with different data

Keep the instruction stable and change only the data. This is useful for generating consistent learning material, product descriptions, support replies, or reports at scale.

In [10]:
lesson_template = PromptTemplate.from_template(
    """Create a mini lesson about {topic}.
Audience: {level}
Include: definition, practical use, and one exercise."""
)

lesson_inputs = [
    {"topic": "Python dictionaries", "level": "beginner"},
    {"topic": "REST APIs", "level": "intermediate"},
    {"topic": "Docker volumes", "level": "beginner"}
]

for item in lesson_inputs:
    formatted = lesson_template.format(**item)
    print("\\n---", item["topic"], "---")
    print(llm.invoke(formatted).content)

\n--- Python dictionaries ---
## Mini Lesson: Python Dictionaries  
*Target audience: absolute beginners*  

---

### 1️⃣ What is a Dictionary?

| Concept | Explanation |
|---------|-------------|
| **Name** | *Dictionary* (also called **dict**) |
| **What it is** | A **collection of key‑value pairs**. Think of it like a real‑world dictionary: you look up a **word** (the *key*) and you get its **definition** (the *value*). |
| **Syntax** | `{ key1: value1, key2: value2, … }` |
| **Key rules** | • Keys must be **unique** and **immutable** (e.g., strings, numbers, tuples). <br>• Values can be anything (numbers, strings, lists, other dicts, …). |
| **Why it’s useful** | Quickly retrieve data by a meaningful name instead of by position (as you would with a list). Perfect for representing structured information like a person’s profile, a product catalog, configuration settings, etc. |

**Simple visual analogy**

```
real‑world dictionary
   "apple"  →  "a round fruit that is red or green"
 

## 10. Partial variables versus normal variables

- A **normal variable** is supplied each time you format or invoke the template.
- A **partial variable** is fixed once and reused, such as a company role, tone, or standard policy.

Use partial variables when part of the prompt remains constant across many requests.

In [12]:
base_template = PromptTemplate.from_template(
    "You are a {role}. Use a {tone} tone. Explain {topic} with one example."
)

devops_template = base_template.partial(
    role="Cloud DevOps mentor",
    tone="clear and practical"
)

print(devops_template.format(topic="Infrastructure as Code"))

You are a Cloud DevOps mentor. Use a clear and practical tone. Explain Infrastructure as Code with one example.


## 11. Template design tips and common mistakes

- Use meaningful placeholder names: `{student_level}` is clearer than `{x}`.
- Keep instructions separate from user data.
- Do not forget required variables.
- Escape literal braces as `{{` and `}}` when the prompt must display braces.
- A `PromptTemplate` formats text; it does not call the model unless you connect it in a chain.
- Validate templates during development and test edge cases such as empty strings.

In [13]:
brace_template = PromptTemplate.from_template(
    "Return a Python dictionary like {{'status': 'ok'}} for topic: {topic}"
)
print(brace_template.format(topic="database connection"))

Return a Python dictionary like {'status': 'ok'} for topic: database connection


## Day 2 completion checklist

- [ ] I can explain the concept without reading the notebook.
- [ ] I understand why and when it is useful.
- [ ] I can modify the example for a different IT topic.
- [ ] I can identify at least one limitation or failure case.
- [ ] I completed the practice task.

**Recommended workflow:** Read the explanation → run the example → change the input → compare outputs → explain the result in your own words.